# DNN Export — ResidualMLP on Raw Snapshot Features

In [ ]:
import json
import os
import sys

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, "../..")
from polybot.adapters.dnn_models import ResidualMLP

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_COLS = [
    "btc_price",
    "elapsed_pct",
    "market_volume",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
]

MODELS_DIR = "../../models"
DATA_DIR = "../../data"

print(f"Features: {len(RAW_COLS)} raw snapshot features")

## 1. Load Data & Split

In [ ]:
# Load all snapshots from pre-built features (no SQLite needed)
rows = []
with open("../../data/latest_features.jsonl") as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df[RAW_COLS] = df[RAW_COLS].fillna(0)

# 80/20 split by candle (time-ordered)
candle_ids = df["candle_id"].unique()
split_idx = int(len(candle_ids) * 0.8)
train_ids = set(candle_ids[:split_idx])
val_ids = set(candle_ids[split_idx:])

df_train = df[df["candle_id"].isin(train_ids)].copy()
df_val = df[df["candle_id"].isin(val_ids)].copy()

X_train_raw = df_train[RAW_COLS].values.astype(np.float32)
y_train = df_train["target"].values.astype(np.float32)
X_val_raw = df_val[RAW_COLS].values.astype(np.float32)
y_val = df_val["target"].values.astype(np.float32)

print(f"Train: {len(train_ids)} candles ({len(df_train):,} snapshots)")
print(f"Val:   {len(val_ids)} candles ({len(df_val):,} snapshots)")

## 2. Train ResidualMLP

In [ ]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)

# Build model
model = ResidualMLP(input_size=len(RAW_COLS))
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ResidualMLP: {params:,} params ({len(RAW_COLS)} features)")

# Training setup
EPOCHS = 50
PATIENCE = 8
BATCH_SIZE = 256
LR = 0.001

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = nn.BCEWithLogitsLoss()

X_t = torch.tensor(X_train, dtype=torch.float32)
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=True)

# Early stopping on a held-out slice of training data
sp = int(len(X_train) * 0.9)
X_es = torch.tensor(X_train[sp:], dtype=torch.float32)
y_es = torch.tensor(y_train[sp:], dtype=torch.float32).unsqueeze(1)

best_vl, best_state, no_improve = float("inf"), None, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    for bx, by in loader:
        optimizer.zero_grad()
        loss_fn(model(bx), by).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    scheduler.step()
    model.eval()
    with torch.no_grad():
        vl = loss_fn(model(X_es), y_es).item()
    marker = ""
    if vl < best_vl:
        best_vl = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
        marker = " *"
    else:
        no_improve += 1
    if epoch % 5 == 0 or marker:
        print(f"  Epoch {epoch:2d}: val_loss={vl:.4f}  lr={scheduler.get_last_lr()[0]:.6f}{marker}")
    if no_improve >= PATIENCE:
        print(f"  Early stop at epoch {epoch}")
        break

if best_state:
    model.load_state_dict(best_state)
model.eval()
print(f"\nBest val_loss: {best_vl:.4f}")

## 3. Calibrate & Export

In [ ]:
# Generate raw probabilities on validation set
X_v = torch.tensor(X_val, dtype=torch.float32)
model.eval()
with torch.no_grad():
    probs_raw = torch.sigmoid(model(X_v)).numpy().flatten()

# Isotonic calibration
calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
calibrator.fit(probs_raw, y_val)
probs_cal = calibrator.predict(probs_raw)

acc = accuracy_score(y_val, (probs_cal >= 0.5).astype(int))
brier = brier_score_loss(y_val, probs_cal)
f1 = f1_score(y_val, (probs_cal >= 0.5).astype(int), zero_division=0)

print(f"Raw:        Brier={brier_score_loss(y_val, probs_raw):.4f}")
print(f"Calibrated: Brier={brier:.4f}, Acc={acc * 100:.1f}%, F1={f1 * 100:.1f}%")

# Export model
model_path = os.path.join(MODELS_DIR, "dnn_v1.pt")
torch.save(model, model_path)
print(f"\nModel: {model_path} ({os.path.getsize(model_path) / 1024:.0f} KB)")

# Export scaler
scaler_path = os.path.join(MODELS_DIR, "dnn_scaler_v1.joblib")
joblib.dump(scaler, scaler_path)

# Export calibrator
calibrator_path = os.path.join(MODELS_DIR, "dnn_calibrator_v1.joblib")
joblib.dump(calibrator, calibrator_path)

# Export feature columns
feature_cols_path = os.path.join(MODELS_DIR, "dnn_feature_cols_v1.joblib")
joblib.dump(RAW_COLS, feature_cols_path)
print(f"Feature cols: {len(RAW_COLS)} features")

# Write optimal_features_dnn.json
feature_config = {
    "model": "dnn",
    "features": RAW_COLS,
    "n_features": len(RAW_COLS),
    "architecture": "ResidualMLP",
    "temporal": False,
    "calibrated": True,
    "parameters": params,
    "metrics": {"accuracy": round(acc, 4), "brier": round(brier, 4), "f1": round(f1, 4)},
}
with open(os.path.join(DATA_DIR, "optimal_features_dnn.json"), "w") as f:
    json.dump(feature_config, f, indent=2)
print("All artifacts exported")

## 4. Verify with DnnPredictor

In [ ]:
from polybot.adapters.dnn_predictor import DnnPredictor
from polybot.ports.predictor import Predictor

predictor = DnnPredictor(
    model_path=model_path,
    feature_cols_path=feature_cols_path,
    scaler_path=scaler_path,
    calibrator_path=calibrator_path,
    temporal=False,
)

assert isinstance(predictor, Predictor)

sample_row = {col: float(X_val_raw[0, i]) for i, col in enumerate(RAW_COLS)}
p = predictor.predict(sample_row)
assert 0.0 <= p <= 1.0

print(f"DnnPredictor loaded! temporal=False, {len(RAW_COLS)} features")
print(f"Sample prediction: P(UP) = {p:.4f}")